In [35]:
import pandas as pd 
import numpy as np

In [41]:
df = pd.read_csv("mflix.embedded_movies.csv")

In [ ]:
df.shape

### First look

In [ ]:
df.head()                                                    # top rows

In [ ]:
df.tail()                                                    # bottom rows

In [ ]:
df.sample(min(5, len(df)), random_state=42)                  # random rows; min guards n>len

In [ ]:
# ALTERNATIVES 
df.head(1).T                                                  # transpose for wide data

In [ ]:
df.loc[:5, ['plot','genres[0]']]                         # slice columns of interest

In [25]:
df.isna().sum()

_id                                 0
plot                               80
genres[0]                           0
genres[1]                         105
genres[2]                         706
                                 ... 
plot_embedding_voyage_4_large      80
metacritic                       2221
tomatoes.consensus               2473
tomatoes.website                 2343
tomatoes.boxOffice               2720
Length: 97, dtype: int64

In [27]:
# df.head(100).style.highlight_null(color='salmon')              # inline NaN highlight

### Structural Summary

In [ ]:
df.info(memory_usage='deep')                                  # deep = true bytes of object cols

<class 'pandas.DataFrame'>
RangeIndex: 3483 entries, 0 to 3482
Data columns (total 97 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   _id                            3483 non-null   str    
 1   plot                           3403 non-null   str    
 2   genres[0]                      3483 non-null   str    
 3   genres[1]                      3378 non-null   str    
 4   genres[2]                      2777 non-null   str    
 5   runtime                        3429 non-null   float64
 6   rated                          2415 non-null   str    
 7   cast[0]                        3465 non-null   str    
 8   cast[1]                        3456 non-null   str    
 9   cast[2]                        3443 non-null   str    
 10  cast[3]                        3434 non-null   str    
 11  num_mflix_comments             3483 non-null   int64  
 12  poster                         3169 non-null   str    
 13 

In [29]:
df.info(verbose=False, memory_usage='deep')                   # collapse cols, keep memory

<class 'pandas.DataFrame'>
RangeIndex: 3483 entries, 0 to 3482
Columns: 97 entries, _id to tomatoes.boxOffice
dtypes: float64(12), int64(4), str(81)
memory usage: 104.0 MB


In [31]:
# ALTERNATIVES 
df.dtypes.value_counts()                                      # dtype mix only

str        81
float64    12
int64       4
Name: count, dtype: int64

In [33]:
pd.DataFrame({                                                # dtype + null table
    'dtype'   : df.dtypes,
    'non_null': df.notnull().sum(),
    'null'    : df.isnull().sum()
})

,dtype,non_null,null
_id,str,3483,0
plot,str,3403,80
genres[0],str,3483,0
genres[1],str,3378,105
genres[2],str,2777,706
...,...,...,...
plot_embedding_voyage_4_large,str,3403,80
metacritic,float64,1262,2221
tomatoes.consensus,str,1010,2473
tomatoes.website,str,1140,2343


In [38]:
# ── ERRORS & FIXES ───────────────────────────────────────────

# default memory usage (shallow)    → hides string bloat
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 3483 entries, 0 to 3482
Data columns (total 97 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   _id                            3483 non-null   str    
 1   plot                           3403 non-null   str    
 2   genres[0]                      3483 non-null   str    
 3   genres[1]                      3378 non-null   str    
 4   genres[2]                      2777 non-null   str    
 5   runtime                        3429 non-null   float64
 6   rated                          2415 non-null   str    
 7   cast[0]                        3465 non-null   str    
 8   cast[1]                        3456 non-null   str    
 9   cast[2]                        3443 non-null   str    
 10  cast[3]                        3434 non-null   str    
 11  num_mflix_comments             3483 non-null   int64  
 12  poster                         3169 non-null   str    
 13 

### Column Overview

In [42]:
summary = pd.DataFrame({
    'dtype'   : df.dtypes,                                                # column type
    'non_null': df.notnull().sum(),                                       # present count
    'null'    : df.isnull().sum(),                                        # missing count
    'null_pct': (df.isnull().mean() * 100).round(2),                      # missing %
    'n_unique': df.nunique(),                                             # cardinality
    'sample'  : [                                                         # first non-null
        df[c].dropna().iloc[0] if df[c].notna().any() else None           # guard all-NaN
        for c in df.columns
    ],
}).sort_values('null_pct', ascending=False).copy()                        # copy kills warning
display(summary)

,dtype,non_null,null,null_pct,n_unique,sample
languages[10],str,1,3482,99.97,1,Egyptian (Ancient)
writers[18],str,1,3482,99.97,1,Erdman Penner (story)
writers[21],str,1,3482,99.97,1,Vernon Stallings (story)
writers[24],str,1,3482,99.97,1,Phil Dike (story)
writers[23],str,1,3482,99.97,1,Arthur Heinemann (story)
...,...,...,...,...,...,...
type,str,3483,0,0.00,2,movie
year,str,3483,0,0.00,107,1932
lastupdated,str,3483,0,0.00,3475,2015-09-03 00:32:16.227000000
awards.text,str,3483,0,0.00,729,2 wins.


In [44]:
# ── ALTERNATIVES ─────────────────────────────────────────────
summary_vec = pd.DataFrame({                                              # vectorized
    'dtype'   : df.dtypes,
    'non_null': df.notnull().sum(),
    'null_pct': df.isnull().mean().mul(100).round(2),                     # .mul avoids align
    'n_unique': df.nunique(),
})
summary_vec

,dtype,non_null,null_pct,n_unique
_id,str,3483,0.00,3483
plot,str,3403,2.30,3280
genres[0],str,3483,0.00,20
genres[1],str,3378,3.01,20
genres[2],str,2777,20.27,22
...,...,...,...,...
plot_embedding_voyage_4_large,str,3403,2.30,3296
metacritic,float64,1262,63.77,90
tomatoes.consensus,str,1010,71.00,965
tomatoes.website,str,1140,67.27,1067


In [46]:
summary_vec['sample'] = [
    df[c].dropna().head(1).squeeze() if df[c].notna().any() else None     # squeeze unwraps
    for c in df.columns
]
summary_vec['sample']

_id                                                       573a1392f29313caabcd9ca6
plot                             An ambitious and near insanely violent gangste...
genres[0]                                                                   Action
genres[1]                                                                    Crime
genres[2]                                                                    Drama
                                                       ...                        
plot_embedding_voyage_4_large    JwBR3cq8frivvFlKgT3bk6i8WUoBvQxLTDxp3VQ8I5TGPA...
metacritic                                                                    97.0
tomatoes.consensus               Errol Flynn thrills as the legendary title cha...
tomatoes.website                 http://www.mgm.com/title_title.do?title_star=G...
tomatoes.boxOffice                                                           $8.3k
Name: sample, Length: 97, dtype: object